# end-grad-default-ones-like — worked example 3: end_grad Default on a Batched Per-Sample Loss

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `end-grad-default-ones-like`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A per-sample loss vector of shape `(B,)` has `B` independent gradient flows. If you call `backprop(per_sample_loss)` without an end_grad, the default `ones_like` seed treats each sample's contribution equally — equivalent to computing `per_sample_loss.sum().backward()`. Providing an explicit end_grad of shape `(B,)` with unequal weights is how you implement importance-weighted training.

## Worked solution

We show that the `ones_like` default on a `(B,)` loss is equivalent to `.sum().backward()`, and that an explicit weight vector produces scaled leaf gradients.

**Setup:** We have `x = [x0, x1, x2]` with `requires_grad=True`. The 'per-sample loss' is `loss_i = x_i ** 2`.

**Default path:** seed = `[1, 1, 1]`. Gradient of `x_i^2` is `2*x_i`. Multiplied by seed 1: `dL/dx_i = 2*x_i`. This matches `(x**2).sum().backward()`.

**Weighted path:** seed = `[2, 0.5, 3]`. Now `dL/dx_i = seed_i * 2*x_i`. Element 0 gets double the gradient push, element 1 gets half. This is the correct behavior for importance-weighted loss.

**Both paths produce the same dtype and shape as x** — the shape guarantee is the core of why `ones_like` is the right default.

In [ ]:
import torch as t

t.manual_seed(3)

x = t.tensor([1.0, 2.0, 0.5], requires_grad=True)

# Per-sample loss: L_i = x_i^2
per_sample_loss = x.pow(2)     # shape (3,)
print(f"per_sample_loss: {per_sample_loss.detach().tolist()}")

# Default end_grad = ones_like -> same as .sum().backward()
default_seed = t.ones_like(per_sample_loss)
print(f"Default seed: {default_seed.tolist()}")

# Manually apply chain rule: dL/dx_i = seed_i * 2*x_i
grad_default = default_seed * 2 * x.detach()
print(f"Grad with ones seed: {grad_default.tolist()}")

# Compare to torch autograd on sum loss
x2 = x.detach().requires_grad_(True)
(x2.pow(2).sum()).backward()
print(f"torch.autograd sum backward: {x2.grad.tolist()}")
assert t.allclose(grad_default, x2.grad)

# Explicit weighted seed
weights = t.tensor([2.0, 0.5, 3.0])
grad_weighted = weights * 2 * x.detach()
print(f"Grad with weighted seed {weights.tolist()}: {grad_weighted.tolist()}")

# Compare to torch autograd on weighted sum
x3 = x.detach().requires_grad_(True)
(weights * x3.pow(2)).sum().backward()
print(f"torch.autograd weighted sum backward: {x3.grad.tolist()}")
assert t.allclose(grad_weighted, x3.grad)

print("Default and weighted seeds produce correct leaf gradients.")